# 02 — Build the UBFC-rPPG cache

Turns 32 GB of video into small per-subject signal files plus a table of quality-checked
8-second windows. Everything heavy happens here, once. Later notebooks only read the cache.

Run the **Setup** cell first after every runtime restart.

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/Heartbeat'
!git clone -q https://github.com/Permurez/Heartbeat.git {REPO_DIR} || git -C {REPO_DIR} pull -q
!pip install -q "mediapipe>=0.10.20"

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import importlib, rppg, rppg.pipeline as pipeline
importlib.reload(pipeline)          # pick up code changes after a git pull without restarting
from rppg import Config

cfg = Config()
cfg

## 1. Trial run on one subject

Check the ROI placement on the debug image before processing all 42 subjects.
Green polygons = forehead and cheeks, blue dots = landmarks used for head motion.
ROIs must sit on skin only: no eyebrows, eyes, nostrils, hair or glasses frames.

In [ ]:
import matplotlib.pyplot as plt, os
summary = pipeline.build_cache(cfg, subjects=['subject1'])
display(summary)

img = plt.imread(os.path.join(cfg.cache_dir, 'debug', 'subject1.png'))
plt.figure(figsize=(8, 6)); plt.imshow(img); plt.axis('off'); plt.show()

In [ ]:
# Sanity check: the green channel of the forehead ROI should pulse with the reference PPG
raw = pipeline.load_subject(cfg, 'subject1')
res = pipeline.resample_subject(raw, cfg)
from rppg.groundtruth import bandpass
g = bandpass(res['rgb_lin'][:, 0, 1], cfg.fs, 0.7, 3.0)
p = bandpass(res['ppg'], cfg.fs, 0.7, 3.0)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(res['t'], p / p.std(), label='reference PPG')
ax.plot(res['t'], g / g.std(), label='forehead green (band-passed)', alpha=0.8)
ax.set_xlim(10, 20); ax.legend(); ax.set_xlabel('time [s]'); plt.show()

## 2. Full run (resumable)

Takes roughly 1-2 min per subject. If Colab disconnects, run Setup and this cell again:
cached subjects are skipped.

In [ ]:
summary = pipeline.build_cache(cfg)
summary

## 3. Windows and quality report

In [ ]:
df = pipeline.build_window_table(cfg)
print(f"Windows: {len(df)}, accepted: {df.accepted.sum()} ({df.accepted.mean():.1%} coverage)")
print("\nReject reasons:")
print(df.reject_reasons[df.reject_reasons != ''].str.split(';').explode().value_counts())
per_subject = df.groupby('subject').agg(windows=('accepted', 'size'), accepted=('accepted', 'mean'),
                                        hr_mean=('hr_gt', 'mean'), motion=('motion', 'median'))
per_subject.sort_values('accepted')